# Credit Card Fraud Detection
This project focuses on detecting fraudulent credit card transactions using machine learning techniques. The dataset used for this project is the "Credit Card Fraud Detection" dataset, which contains transactions made by European cardholders in September 2013. The dataset is highly imbalanced, with the majority of transactions being legitimate and a small percentage being fraudulent.

In [9]:
import pandas as pd
import numpy as np

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
from sklearn.metrics import (
    average_precision_score,  # PR-AUC
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [12]:
import sys
print("This notebook is using:", sys.executable)

!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install xgboost lightgbm catboost optuna

This notebook is using: c:\Users\Muhammad Saad Ullah\AppData\Local\Programs\Python\Python313\python.exe


'c:\Users\Muhammad' is not recognized as an internal or external command,
operable program or batch file.
'c:\Users\Muhammad' is not recognized as an internal or external command,
operable program or batch file.


In [13]:
import pip 

In [14]:
import sys
print(sys.executable)

c:\Users\Muhammad Saad Ullah\AppData\Local\Programs\Python\Python313\python.exe


In [15]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.2.0


In [16]:
# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Tuning
import optuna

In [17]:
# Loading the Data 
df = pd.read_csv('creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# Understanding the Data

In [18]:
# Shape (rows, columns)
print(f"Dataset has {df.shape[0]} rows and {df.shape[1]} columns.")

# Column names
print("\nColumns:", df.columns.tolist())

# Data types
print("\nData types:")
print(df.dtypes)

# Summary statistics
df.describe()

Dataset has 284807 rows and 31 columns.

Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Data types:
Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


# Checking for the missing values

In [19]:
# Count missing values
missing = df.isnull().sum() # Here df is the DataFrame we are working with
print(missing[missing > 0])

# Percentage missing
missing_percent = (df.isnull().sum() / len(df)) * 100
print("\nPercentage missing:")
print(missing_percent[missing_percent > 0])

Series([], dtype: int64)

Percentage missing:
Series([], dtype: float64)


## Data Split for Model Training

In [20]:
# df = your loaded dataframe
target_col = "Class"

X = df.drop(columns=[target_col])
y = df[target_col]

# Stratified split keeps fraud ratio similar in train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape, "Fraud rate:", y_train.mean())
print("Val size:  ", X_val.shape,   "Fraud rate:", y_val.mean())

Train size: (227845, 30) Fraud rate: 0.001729245759178389
Val size:   (56962, 30) Fraud rate: 0.0017204452090867595


## Evaluation Function 

In [21]:
def get_scale_pos_weight(y):
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return n_neg / max(n_pos, 1)

scale_pos_weight = get_scale_pos_weight(y_train)
print("scale_pos_weight =", scale_pos_weight)

scale_pos_weight = 577.2868020304569


In [22]:
def evaluate_binary_classifier(y_true, y_proba, threshold=0.5, verbose=True):
    """
    y_true: true labels (0/1)
    y_proba: predicted probability for class 1 (fraud)
    threshold: probability cutoff to turn probability into 0/1 prediction
    """
    y_pred = (y_proba >= threshold).astype(int)

    pr_auc = average_precision_score(y_true, y_proba)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    if verbose:
        tn, fp, fn, tp = cm.ravel()
        print(f"Threshold: {threshold:.3f}")
        print(f"PR-AUC:    {pr_auc:.6f}")
        print(f"Precision: {precision:.6f}")
        print(f"Recall:    {recall:.6f}")
        print(f"F1:        {f1:.6f}")
        print("Confusion matrix [ [TN FP] [FN TP] ]:")
        print(cm)
        print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    return {
        "pr_auc": pr_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "threshold": threshold
    }

## Model Training

## XGBoost Classifier

In [23]:
xgb_model = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

xgb_model.fit(X_train, y_train)

y_val_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]
xgb_metrics = evaluate_binary_classifier(y_val, y_val_proba_xgb, threshold=0.5, verbose=True)
xgb_metrics

Threshold: 0.500
PR-AUC:    0.884983
Precision: 0.863158
Recall:    0.836735
F1:        0.849741
Confusion matrix [ [TN FP] [FN TP] ]:
[[56851    13]
 [   16    82]]
TN=56851, FP=13, FN=16, TP=82


{'pr_auc': 0.8849832706158905,
 'precision': 0.8631578947368421,
 'recall': 0.8367346938775511,
 'f1': 0.8497409326424871,
 'threshold': 0.5}

In [25]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "logloss",
        "scale_pos_weight": scale_pos_weight
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_proba)  # PR-AUC

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=5)

print("Best PR-AUC:", study_xgb.best_value)
print("Best params:", study_xgb.best_params)

[I 2026-03-05 23:43:48,880] A new study created in memory with name: no-name-c74d5358-416c-47c6-b92b-2fcad7d36efc
[I 2026-03-05 23:44:04,954] Trial 0 finished with value: 0.8810142780032387 and parameters: {'n_estimators': 524, 'learning_rate': 0.07284032760435202, 'max_depth': 4, 'subsample': 0.8708577857675681, 'colsample_bytree': 0.5841258614646083, 'min_child_weight': 11, 'reg_lambda': 0.2399570935143414}. Best is trial 0 with value: 0.8810142780032387.
[I 2026-03-05 23:44:37,986] Trial 1 finished with value: 0.8793257669536205 and parameters: {'n_estimators': 1244, 'learning_rate': 0.17498565611476485, 'max_depth': 4, 'subsample': 0.5599755545869616, 'colsample_bytree': 0.6250085063536417, 'min_child_weight': 7, 'reg_lambda': 0.031417499778269985}. Best is trial 0 with value: 0.8810142780032387.
[I 2026-03-05 23:45:43,428] Trial 2 finished with value: 0.8775903619026162 and parameters: {'n_estimators': 1818, 'learning_rate': 0.014001222880463007, 'max_depth': 7, 'subsample': 0.650

Best PR-AUC: 0.8810142780032387
Best params: {'n_estimators': 524, 'learning_rate': 0.07284032760435202, 'max_depth': 4, 'subsample': 0.8708577857675681, 'colsample_bytree': 0.5841258614646083, 'min_child_weight': 11, 'reg_lambda': 0.2399570935143414}


In [26]:
best_xgb = XGBClassifier(
    **study_xgb.best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

best_xgb.fit(X_train, y_train)

y_val_proba_best_xgb = best_xgb.predict_proba(X_val)[:, 1]
evaluate_binary_classifier(y_val, y_val_proba_best_xgb, threshold=0.5, verbose=True)

Threshold: 0.500
PR-AUC:    0.881014
Precision: 0.840000
Recall:    0.857143
F1:        0.848485
Confusion matrix [ [TN FP] [FN TP] ]:
[[56848    16]
 [   14    84]]
TN=56848, FP=16, FN=14, TP=84


{'pr_auc': 0.8810142780032387,
 'precision': 0.84,
 'recall': 0.8571428571428571,
 'f1': 0.8484848484848485,
 'threshold': 0.5}

## LightGBM Classifier

In [27]:
lgbm_model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

lgbm_model.fit(X_train, y_train)

y_val_proba_lgbm = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_metrics = evaluate_binary_classifier(y_val, y_val_proba_lgbm, threshold=0.5, verbose=True)
lgbm_metrics

[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014676 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001729 -> initscore=-6.358339
[LightGBM] [Info] Start training from score -6.358339
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'pr_auc': 0.7491021036407204,
 'precision': 0.7575757575757576,
 'recall': 0.7653061224489796,
 'f1': 0.7614213197969543,
 'threshold': 0.5}

In [28]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 6000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": scale_pos_weight
    }

    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_proba)

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=5)

print("Best PR-AUC:", study_lgbm.best_value)
print("Best params:", study_lgbm.best_params)

[I 2026-03-05 23:48:16,506] A new study created in memory with name: no-name-e5049d40-5c27-4df7-8b8d-fe7a32bdeac7


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.051567 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001729 -> initscore=-6.358339
[LightGBM] [Info] Start training from score -6.358339
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

[I 2026-03-05 23:51:28,472] Trial 0 finished with value: 0.8835720380889038 and parameters: {'n_estimators': 5637, 'learning_rate': 0.005196675675014303, 'num_leaves': 114, 'max_depth': 10, 'min_child_samples': 159, 'subsample': 0.5050575479781696, 'colsample_bytree': 0.6363366184495479, 'reg_lambda': 0.007754604705334246}. Best is trial 0 with value: 0.8835720380889038.


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011426 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001729 -> initscore=-6.358339
[LightGBM] [Info] Start training from score -6.358339
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2026-03-05 23:52:49,656] Trial 1 finished with value: 0.8821763464708268 and parameters: {'n_estimators': 2261, 'learning_rate': 0.006389039778415082, 'num_leaves': 40, 'max_depth': 10, 'min_child_samples': 152, 'subsample': 0.7803130575368482, 'colsample_bytree': 0.8487698200406746, 'reg_lambda': 0.8252066135709856}. Best is trial 0 with value: 0.8835720380889038.


...

[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043918 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001729 -> initscore=-6.358339
[LightGBM] [Info] Start training from score -6.358339
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

[I 2026-03-05 23:54:21,928] Trial 3 finished with value: 0.06141825027443805 and parameters: {'n_estimators': 5327, 'learning_rate': 0.08976697666934576, 'num_leaves': 167, 'max_depth': 3, 'min_child_samples': 72, 'subsample': 0.8440409592799815, 'colsample_bytree': 0.9812947246884779, 'reg_lambda': 0.0025295692956297353}. Best is trial 0 with value: 0.8835720380889038.


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042349 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001729 -> initscore=-6.358339
[LightGBM] [Info] Start training from score -6.358339
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

[I 2026-03-05 23:54:35,087] Trial 4 finished with value: 0.12512647212577652 and parameters: {'n_estimators': 928, 'learning_rate': 0.03886916149786914, 'num_leaves': 240, 'max_depth': 3, 'min_child_samples': 19, 'subsample': 0.530124326226467, 'colsample_bytree': 0.899877781742882, 'reg_lambda': 0.037058616832807104}. Best is trial 0 with value: 0.8835720380889038.


Best PR-AUC: 0.8835720380889038
Best params: {'n_estimators': 5637, 'learning_rate': 0.005196675675014303, 'num_leaves': 114, 'max_depth': 10, 'min_child_samples': 159, 'subsample': 0.5050575479781696, 'colsample_bytree': 0.6363366184495479, 'reg_lambda': 0.007754604705334246}


In [29]:
best_lgbm = LGBMClassifier(
    **study_lgbm.best_params,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

best_lgbm.fit(X_train, y_train)

y_val_proba_best_lgbm = best_lgbm.predict_proba(X_val)[:, 1]
evaluate_binary_classifier(y_val, y_val_proba_best_lgbm, threshold=0.5, verbose=True)

[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010752 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001729 -> initscore=-6.358339
[LightGBM] [Info] Start training from score -6.358339
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'pr_auc': 0.8835720380889038,
 'precision': 0.9222222222222223,
 'recall': 0.8469387755102041,
 'f1': 0.8829787234042553,
 'threshold': 0.5}

## Catboost Classifier

In [30]:
cat_model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    verbose=False,
    class_weights=[1.0, scale_pos_weight]
)

cat_model.fit(X_train, y_train)

y_val_proba_cat = cat_model.predict_proba(X_val)[:, 1]
cat_metrics = evaluate_binary_classifier(y_val, y_val_proba_cat, threshold=0.5, verbose=True)
cat_metrics

Threshold: 0.500
PR-AUC:    0.870610
Precision: 0.770642
Recall:    0.857143
F1:        0.811594
Confusion matrix [ [TN FP] [FN TP] ]:
[[56839    25]
 [   14    84]]
TN=56839, FP=25, FN=14, TP=84


{'pr_auc': 0.8706096278638613,
 'precision': 0.7706422018348624,
 'recall': 0.8571428571428571,
 'f1': 0.8115942028985508,
 'threshold': 0.5}

In [32]:
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 800, 6000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 30.0, log=True),
        "random_seed": 42,
        "verbose": False,
        "class_weights": [1.0, scale_pos_weight]
    }

    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_proba)

study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=1)

print("Best PR-AUC:", study_cat.best_value)
print("Best params:", study_cat.best_params)

[I 2026-03-06 00:32:02,675] A new study created in memory with name: no-name-1e254598-d334-49be-98d2-7cabd81fdf77
[I 2026-03-06 00:36:17,345] Trial 0 finished with value: 0.8183109334344829 and parameters: {'iterations': 5974, 'learning_rate': 0.09081433568235098, 'depth': 4, 'l2_leaf_reg': 0.005557733718874975}. Best is trial 0 with value: 0.8183109334344829.


Best PR-AUC: 0.8183109334344829
Best params: {'iterations': 5974, 'learning_rate': 0.09081433568235098, 'depth': 4, 'l2_leaf_reg': 0.005557733718874975}


In [33]:
best_cat = CatBoostClassifier(
    **study_cat.best_params,
    random_seed=42,
    verbose=False,
    class_weights=[1.0, scale_pos_weight]
)

best_cat.fit(X_train, y_train)

y_val_proba_best_cat = best_cat.predict_proba(X_val)[:, 1]
evaluate_binary_classifier(y_val, y_val_proba_best_cat, threshold=0.5, verbose=True)

Threshold: 0.500
PR-AUC:    0.818311
Precision: 0.840426
Recall:    0.806122
F1:        0.822917
Confusion matrix [ [TN FP] [FN TP] ]:
[[56849    15]
 [   19    79]]
TN=56849, FP=15, FN=19, TP=79


{'pr_auc': 0.8183109334344829,
 'precision': 0.8404255319148937,
 'recall': 0.8061224489795918,
 'f1': 0.8229166666666666,
 'threshold': 0.5}

## Report
# Day 6 — Model Comparison Report (Fraud Detection)

## Setup (what these scores mean)
- Task: **Binary classification** (Fraud = 1, Not Fraud = 0)
- Dataset: **Highly imbalanced** (fraud is rare)
- Threshold used for all models: **0.5** (probability ≥ 0.5 → predict Fraud)

### Metrics (simple meaning)
- **PR-AUC**: Overall score for imbalanced problems (higher is better).
- **Precision**: When the model says “Fraud”, how often it is correct (higher = fewer false alarms).
- **Recall**: Out of all real fraud cases, how many we catch (higher = fewer missed frauds).
- **F1**: A balance between precision and recall (higher is better).

---

## Results Summary (at threshold = 0.5)

| Model | PR-AUC | Precision | Recall | F1 |
|------|-------:|----------:|-------:|---:|
| **XGBoost** | 0.8810 | 0.8400 | **0.8571** | 0.8485 |
| **LightGBM** | **0.8836** | **0.9222** | 0.8469 | **0.8830** |
| **CatBoost** | 0.8183 | 0.8404 | 0.8061 | 0.8229 |

---

## Comparison (what we learn from this)

### 1) Best overall model in this run: **LightGBM**
LightGBM has:
- **Highest PR-AUC (0.8836)** → best overall performance for this imbalanced dataset.
- **Highest Precision (0.9222)** → it produces the fewest false fraud alerts among the three.
- **Highest F1 (0.8830)** → best balance between catching fraud and not flagging too many normal transactions.

**Interpretation:**  
If your goal is a strong overall fraud model with fewer “false alarms”, **LightGBM performed best**.

---

### 2) Best model for catching the most fraud: **XGBoost (slightly)**
XGBoost has:
- **Highest Recall (0.8571)** → it catches slightly more fraud cases than LightGBM (0.8469).

**Interpretation:**  
If your main goal is “miss as few fraud cases as possible”, **XGBoost is marginally better** here, but the difference is small.

---

### 3) CatBoost underperformed in this run
CatBoost has:
- Lower PR-AUC (0.8183)
- Lower recall (0.8061)
- Lower F1 (0.8229)

**Interpretation:**  
With the current setup, **CatBoost is not the top choice**. This can happen because:
- CatBoost shines more when there are strong *category-type columns* (like city/job/card_type).
- This dataset (the common credit card fraud dataset) is mostly numeric engineered features, so CatBoost may not have its usual advantage.
- It may improve with tuning, but as-is it is behind XGBoost/LightGBM.

---

## Final takeaway (simple)
- **LightGBM is the current winner** overall (best PR-AUC, precision, and F1).
- **XGBoost is a close second**, and it catches *slightly more* fraud (best recall).
- **CatBoost is currently third** for this dataset with these settings.

---

## Next steps (recommended)
1. **Tune hyperparameters (Optuna)** for the top 2 models first:
   - LightGBM and XGBoost
2. **Try threshold adjustment**:
   - If you want to catch more fraud, try thresholds like `0.3`, `0.2`, `0.1`
   - Compare precision vs recall at each threshold
3. Pick the final model based on your goal:
   - Prefer **higher recall** (catch more fraud) → lean XGBoost / lower threshold
   - Prefer **higher precision** (fewer false alarms) → LightGBM already looks best

## Saving the best model

In [34]:
import joblib
joblib.dump(best_lgbm, "best_fraud_detection_model.joblib")
print("Best model saved as best_fraud_detection_model.joblib")

Best model saved as best_fraud_detection_model.joblib
